In [ ]:
import dask
import glob
import numpy as np
import xarray as xr
import geopandas as gpd
import cartopy.crs as ccrs
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from pyproj import Transformer
import matplotlib.colors as mcolors
from shapely.geometry import Polygon, Point
import cartopy.feature as cfeature

In [ ]:
def plot_hudson_bay(west, south, ds):
    # Define projections
    crs = ccrs.Stereographic(
        central_latitude=90,
        central_longitude=-45
    )

    crs_nsidc = ccrs.Stereographic(
        central_latitude=90,
        central_longitude=-45,
        true_scale_latitude=70,
        false_easting=0,
        false_northing=0
    )

    # Define levels (boundaries for the colors)
    levels = np.linspace(0, 100, 11)
    norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=plt.cm.Blues_r.N, clip=True)

    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw={'projection': crs})

    # Ocean
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), zorder=0, facecolor='#08306B')

    # Sea ice concentration
    im = ax.pcolormesh(
        ds.x.values,
        ds.y.values,
        west.values,
        cmap='Blues_r',
        norm=norm,
        transform=crs_nsidc,
        zorder=1
    )

    im = ax.pcolormesh(
        ds.x.values,
        ds.y.values,
        south.values,
        cmap='Blues_r',
        norm=norm,
        transform=crs_nsidc,
        zorder=1
    )

    # Land
    ax.add_feature(cfeature.LAND.with_scale('50m'), zorder=2, facecolor='gray', edgecolor='black', linewidth=0.5)

    # Coastlines
    ax.coastlines(resolution='50m', zorder=3)

    # Approximate extent around Hudson Bay in degrees
    hudson_extent = [-87, -80, 49, 67]
    ax.set_extent(hudson_extent, crs=ccrs.PlateCarree())

    # Gridlines
    ax.gridlines(draw_labels=False, linewidth=0.5, alpha=0.5, color='white')

    # West and South Hudson Bay boxes
    west_bounds = {'lon_min': -95, 'lon_max': -88, 'lat_min': 56, 'lat_max': 63}
    south_bounds = {'lon_min': -88, 'lon_max': -75, 'lat_min': 51, 'lat_max': 59}

    draw_box(ax, west_bounds, edgecolor='red', linewidth=2)
    draw_box(ax, south_bounds, edgecolor='red', linewidth=2)

    ax.text(-90, 60, "West", transform=ccrs.PlateCarree(), fontsize=12, color='red', ha='center', va='center', fontweight='bold')
    ax.text(-85, 54, "South", transform=ccrs.PlateCarree(), fontsize=12, color='red', ha='center', va='center', fontweight='bold')

    # Colorbar with percentage formatting
    cbar = plt.colorbar(im, orientation='vertical', boundaries=levels, ticks=levels, pad=0.05, shrink=0.7)
    cbar.set_label('Sea Ice Concentration (%)', fontsize=12)

    # Title in same style
    date_str = ds["time"].dt.strftime("%d %b %Y").item()
    ax.set_title(f'Sea Ice Concentration, {date_str}', fontsize=16, pad=20)

    plt.tight_layout()
    plt.show()



def draw_box(ax, bounds, edgecolor='red', linewidth=2):
    """
    Draw a rectangular box on a Cartopy map.
    
    Parameters:
    - ax: matplotlib axis with cartopy projection
    - bounds: dict with keys 'lon_min', 'lon_max', 'lat_min', 'lat_max'
    """
    # Corners of the box in lon/lat
    lons = [bounds['lon_min'], bounds['lon_max'], bounds['lon_max'], bounds['lon_min'], bounds['lon_min']]
    lats = [bounds['lat_min'], bounds['lat_min'], bounds['lat_max'], bounds['lat_max'], bounds['lat_min']]
    
    # Plot the box using PlateCarree projection (geodetic)
    ax.plot(lons, lats, transform=ccrs.PlateCarree(), color=edgecolor, linewidth=linewidth)    

In [ ]:
# Load data
AMSR2_12km = xr.open_dataset('/glade/work/skygale/pbi-data/metrics-data/AMSR2_12.5km_metrics.nc')
AMSR2_25km = xr.open_dataset('/glade/work/skygale/pbi-data/metrics-data/AMSR2_25km_metrics.nc')
G02202_25km = xr.open_dataset('/glade/work/skygale/pbi-data/metrics-data/G02202_25km_metrics.nc')

# Print variable names to verify
for vars in G02202_25km.data_vars:
    print(vars)

# Combine datasets into a list for easier handling
ds_all = [AMSR2_12km, AMSR2_25km, G02202_25km]

In [ ]:
# Plot time series
regions = ['west', 'south']
variables = ['freeze_up_day', 'ice_free_days', 'breakup_day']
titles = {
    'freeze_up_day': 'Freeze-Up Day',
    'ice_free_days': 'Ice-Free Days',
    'breakup_day': 'Breakup Day'
}
    
fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex=True)

for i, var in enumerate(variables):
    for j, region in enumerate(regions):
        ax = axes[i, j]
        ax.grid(True)

        for ds in ds_all:
            data = ds[f"{region}_{var}"]
        
            # Plot each year as a separate line
            for year in data['year'].values:
                yearly_data = data.sel(year=year)
                ax.plot(yearly_data['time'].dt.year, yearly_data, label=str(year))
        
        # Set titles as regions on top of plots
        if i == 0:
            ax.set_title(f"{region.capitalize()} Hudson Bay")

        # Set y axis label as variable for left plots
        if j == 0:
            ax.set_ylabel(titles[var])

plt.tight_layout()
plt.show()